# Celltypist and cellhint

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd

## rna

In [ ]:
adata_rna_ann = sc.read_h5ad('../5_matrix_merge/processed/ann_cell/ann_cell_rna.h5ad')
adata_rna_ann

In [ ]:
adata_rna_no_ann = sc.read_h5ad('../5_matrix_merge/processed/no_annotation/filter_cell_rna.h5ad')
adata_rna_no_ann

In [ ]:
adata_rna_no_ann.obs = adata_rna_no_ann.obs[['dataset','barcode', 'sample']]

In [ ]:
del adata_rna_no_ann.var

In [ ]:
adata_rna_no_ann.obs['cellType'] = 'notAvailable'

In [ ]:
adata_rna_no_ann

In [ ]:
adata_rna = ad.concat([adata_rna_ann, adata_rna_no_ann], join='outer', merge='unique', axis=0)

In [ ]:
adata_rna

In [ ]:
adata_rna.write_h5ad('./processed/adata_rna.h5ad', compression='gzip')

## 10x to ann

In [ ]:
adata = sc.read_h5ad('./processed/adata_rna.h5ad')
adata

In [ ]:
adata.X

In [ ]:
adata[:1000000,:].X

In [ ]:
1671796325/2147483647

In [ ]:
def anndata_to_10x(adata:sc.AnnData, _dir):
    from scipy.io import mmwrite
    from os.path import join, exists
    from os import makedirs
    if not exists(_dir):
        makedirs(_dir)
    X = adata[:1000000,:].X.T.tocoo()
    mmwrite(join(_dir, 'matrix.mtx'), X)
    var = pd.DataFrame(adata[:1000000,:].var.index, columns=['gene_symbol'])
    var.to_csv(join(_dir, 'genes.tsv'), sep='\t', header=True, index=False)
    adata[:1000000,:].obs.to_csv(join(_dir, 'barcodes.tsv'), sep='\t', header=True, index=True)

anndata_to_10x(adata, './processed/scanpy_10x/part_1/')

In [ ]:
adata[1000000:,:].X

In [ ]:
1442925164/2147483647

In [ ]:
1442925164+1671796325

In [ ]:
def anndata_to_10x(adata:sc.AnnData, _dir):
    from scipy.io import mmwrite
    from os.path import join, exists
    from os import makedirs
    if not exists(_dir):
        makedirs(_dir)
    X = adata[1000000:,:].X.T.tocoo()
    mmwrite(join(_dir, 'matrix.mtx'), X)
    var = pd.DataFrame(adata[1000000:,:].var.index, columns=['gene_symbol'])
    var.to_csv(join(_dir, 'genes.tsv'), sep='\t', header=True, index=False)
    adata[1000000:,:].obs.to_csv(join(_dir, 'barcodes.tsv'), sep='\t', header=True, index=True)

anndata_to_10x(adata, './processed/scanpy_10x/part_2/')

## atac

In [ ]:
adata_atac = sc.read_h5ad('../5_matrix_merge/processed/all_cell/atac.h5ad')
adata_atac

In [ ]:
adata_rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [ ]:
adata_atac = adata_atac[adata_rna.obs_names.tolist()]
adata_atac

In [ ]:
adata_atac.obs['cellType'] = adata_rna.obs['cellType'].tolist()

In [ ]:
adata_atac

In [ ]:
adata_atac.write_h5ad('./processed/adata_atac.h5ad', compression='gzip')

## celltypist_cellhint

In [ ]:
# import celltypist
# import cellhint
# import os
# import scanpy as sc
# os.chdir('/path/to/hemelas_project/z_others/6_ann_batch')

# adata = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(adata, target_sum = 1e4)
# sc.pp.log1p(adata)

# adata = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True).to_adata()

# adata.write_h5ad('./processed/adata_rna_celltypist.h5ad', compression='gzip')

# cellhint.integrate(adata, 'sample', 'majority_voting')

# sc.tl.umap(adata)

# adata.write_h5ad('./processed/adata_rna_celltypist_cellhint.h5ad', compression='gzip')

In [ ]:
test = sc.read_h5ad('./processed/adata_rna_celltypist.h5ad')

In [ ]:
test

In [ ]:
adata = sc.read_h5ad('./processed/adata_rna_celltypist_cellhint.h5ad')
adata

In [ ]:
adata.obs.to_csv('processed/cellTypist.csv')

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

# # Add annotation levels

In [ ]:
import os
os.chdir('/path/to/hemelas_project/z_others/')

## cellhint majority_voting

In [ ]:
data = sc.read_h5ad('6_ann_batch/processed/adata_rna_celltypist_cellhint.h5ad')
data

In [ ]:
cell_type = pd.read_csv('./6_ann_batch/processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell_type

In [ ]:
all = pd.merge(data.obs, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = data.obs_names.tolist()
all

In [ ]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
data.write_h5ad('6_ann_batch/processed/adata_rna_celltypist_cellhint.h5ad', compression='gzip')

## cellhint predicted_labels

In [ ]:
data = sc.read_h5ad('6_ann_visual/processed/adata_rna_cellhint_1.h5ad')
data

In [ ]:
cell_type = pd.read_csv('./6_ann_visual/processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell_type

In [ ]:
all = pd.merge(data.obs, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = data.obs_names.tolist()
all

In [ ]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
data.obs['level_3'].value_counts()

In [ ]:
data.obs['level_2'].value_counts()

In [ ]:
data.obs['level_1'].value_counts()

# snapatac2

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import pandas as pd
import muon

## rna

### spectral 1

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_rna_snap_umap.h5ad')

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna_snap_umap.h5ad')
rna

In [ ]:
rna.obsm['X_spectral'].dtype

In [ ]:
sc.pl.umap(rna, color="level_1")

### spectral 2

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]

# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_rna_snap2_umap.h5ad')

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna_snap2_umap.h5ad')
rna

In [ ]:
import harmonypy as hm

In [ ]:
meta_data = rna.obs
vars_use = ['sample']
data_mat = rna.obsm['X_spectral']
ho = hm.run_harmony(data_mat, meta_data, vars_use, max_iter_harmony=20)

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='technology')

## norm

In [ ]:
# adata = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# adata.raw = adata
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, svd_solver='arpack')

# sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
# sc.tl.umap(adata)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = adata.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = adata.obs_names.tolist()
# all

# adata.obs = all

# adata.write_h5ad('./processed/adata_rna_umap.h5ad')

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna_umap.h5ad')
rna

In [ ]:
pca = pd.DataFrame(rna.obsm['X_pca'])
pca

In [ ]:
pca.to_csv('./processed/rna_pca.csv')

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="technology")

## atac 

### spectral

In [ ]:
# rna = sc.read_h5ad('./processed/adata_atac.h5ad')
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
# data = pd.read_csv('./visual/sample_all.csv', index_col=0)
# data['sample_'] = data.index.tolist()
# all = pd.merge(cell, data, left_on='sample', right_on='sample_')
# all.index = rna.obs_names.tolist()
# all

# cell_type = pd.read_csv('./processed/cellType_level.csv')
# cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
# all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
# all.index = rna.obs_names.tolist()
# all

# rna.obs = all

# rna.write_h5ad('./processed/adata_atac_umap.h5ad')

In [ ]:
atac = sc.read_h5ad('./processed/adata_atac_umap.h5ad')
atac

In [ ]:
sc.pl.umap(atac, color="level_2")

In [ ]:
sc.pl.umap(atac, color="technology")

### ti-idf

In [ ]:
atac = sc.read_h5ad('./processed/adata_atac.h5ad')
atac

In [ ]:
muon.atac.tl.lsi(atac)
atac

In [ ]:
embedding = np.load('./processed/atac_lsi.npy')
embedding

In [ ]:
b = pd.read_

In [ ]:
a = pd.DataFrame(embedding)
a

In [ ]:
a.to_csv('./processed/atac_lsi.csv')

## joint

### spectral

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)

In [ ]:
# atac = sc.read_h5ad('./processed/adata_atac.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)

In [ ]:
# assert (rna.obs_names == atac.obs_names).all()
# embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
# np.save('embedding.npy', embedding)

# rna = sc.read_h5ad('./processed/adata_rna.h5ad')
# embedding = np.load('./processed/embedding.npy')
# rna.obsm['rna_atac'] = embedding
# snap.tl.umap(rna, use_rep='rna_atac')
# np.save('./processed/umap.npy', rna.obsm['X_umap'])

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [ ]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

In [ ]:
rna.obs = all

In [ ]:
umap = np.load('./processed/umap.npy')

In [ ]:
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="dataset")

In [ ]:
sc.pl.umap(rna, color="technology")

### spectral 2

In [ ]:
# rna = sc.read_h5ad('./processed/adata_rna.h5ad')

# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]

# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')

In [ ]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

In [ ]:
rna.obs = all

In [ ]:
umap = np.load('./processed/umap_snap2.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="technology")

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="dataset")

### norm 

In [ ]:
atac = sc.read_h5ad('./processed/adata_atac_umap.h5ad')

rna = sc.read_h5ad('./processed/adata_rna_umap.h5ad')

In [ ]:
rna.obsm['X_spectral'] = rna.obsm['X_pca'][:, :30].astype(np.float64)

In [ ]:
rna.obsm['X_spectral']

In [ ]:
atac.obsm['X_spectral']

In [ ]:
embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]

## harmony

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(rna, 'sample')

In [ ]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()

cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
all

In [ ]:
rna.obs = all

In [ ]:
umap = np.load('./processed/umap_harmony.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="level_1")

In [ ]:
sc.pl.umap(rna, color="technology")

## cellhint

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

In [ ]:
cell = pd.read_csv('./processed/cellTypist.csv', index_col=0)
data = pd.read_csv('./visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = rna.obs_names.tolist()
cell_type = pd.read_csv('./processed/cellType_level.csv')
cell_type = cell_type.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
all = pd.merge(all, cell_type, left_on='predicted_labels', right_on='level_3')
all.index = rna.obs_names.tolist()
rna.obs = all

In [ ]:
umap = np.load('./processed/umap_cellhint.npy')
rna.obsm['X_umap'] = umap

In [ ]:
sc.pl.umap(rna, color="level_2")

In [ ]:
sc.pl.umap(rna, color="technology")

# Multiple algorithms and visual

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

## spectral

### rna_raw

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
embedding = pd.DataFrame(rna.obsm['X_spectral'])
embedding

In [ ]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/rna/embedding_raw.csv')

In [ ]:
umap = pd.DataFrame(rna.obsm['X_umap'])
umap

In [ ]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/rna/umap_raw.csv')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### rna_harmony

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/rna/umap_harmony.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_raw

In [ ]:
rna = sc.read_h5ad('processed/adata_atac_spectral_umap.h5ad')
rna

In [ ]:
embedding = pd.DataFrame(rna.obsm['X_spectral'])
embedding

In [ ]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/atac/embedding_raw.csv')

In [ ]:
umap = pd.DataFrame(rna.obsm['X_umap'])
umap

In [ ]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/atac/umap_raw.csv')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_harmony

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/atac/umap_harmony.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_raw

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_snap2.npy'))
embedding

In [ ]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_raw.csv')

In [ ]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_snap2.npy'))
umap

In [ ]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_raw.csv')

In [ ]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_snap2.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_multi_harmony

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_harmony.npy'))
embedding

In [ ]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_multi_harmony.csv')

In [ ]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_harmony.npy'))
umap

In [ ]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_multi_harmony.csv')

In [ ]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_harmony.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_harmony_multi

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('processed/z_embedding_umap/multi_spectral/joint/umap_harmony_multi.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### joint_cellhint

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
embedding = pd.DataFrame(np.load('processed/snapatac2/embedding_cellhint.npy'))
embedding

In [ ]:
embedding.to_csv('processed/z_embedding_umap/multi_spectral/joint/embedding_cellhint.csv')

In [ ]:
umap = pd.DataFrame(np.load('processed/snapatac2/umap_cellhint.npy'))
umap

In [ ]:
umap.to_csv('processed/z_embedding_umap/multi_spectral/joint/umap_cellhint.csv')

In [ ]:
rna.obsm['X_umap'] = np.load('processed/snapatac2/umap_cellhint.npy')

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

## wnn

### rna_raw

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('./processed/pca/rna_pca_umap.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### rna_harmony

In [ ]:
umap = pd.read_csv('./processed/pca/rna_pca_harmony_umap.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_raw

In [ ]:
umap = pd.read_csv('./processed/lsi/atac_lsi_umap.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### atac_harmony

In [ ]:
umap = pd.read_csv('./processed/lsi/atac_lsi_harmony_umap.csv', index_col=0)
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

### atac_new

In [ ]:
data = sc.read_h5ad('./processed/adata_atac_lsi_harmony.h5ad')
data

In [ ]:
# data.obsm['X_lsi'][:,1:])
# data.obsm['X_pca_harmony']

In [ ]:
sc.pl.umap(data, color='dataset')

In [ ]:
data.obsm['X_umap'] = data.obsm['X_lsi_umap']

In [ ]:
sc.pl.umap(data, color='dataset')

### wnn_multi

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('./processed/wnn/umap_m.csv')
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

### wnn_harmony_multi

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')
rna

In [ ]:
umap = pd.read_csv('./processed/wnn/umap_h_m.csv')
umap

In [ ]:
rna.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna, color='level_2')

In [ ]:
sc.pl.umap(rna, color='dataset')

In [ ]:
sc.pl.umap(rna, color='technology')

# benchmarking

In [ ]:
import scanpy as sc
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

# Compute

In [ ]:
rna = sc.read_h5ad('processed/adata_rna_spectral_umap.h5ad')

In [ ]:
rna.obsm['RNA_spectral'] = rna.obsm['X_spectral']

embedding = pd.read_csv('processed/multi_spectral/rna/embedding_harmony.csv', index_col=0).to_numpy()
rna.obsm['RNA_spectral_harmony'] = embedding

embedding = pd.read_csv('processed/multi_spectral/atac/embedding_raw.csv', index_col=0).to_numpy()
rna.obsm['ATAC_spectral'] = embedding

embedding = pd.read_csv('processed/multi_spectral/atac/embedding_harmony.csv', index_col=0).to_numpy()
rna.obsm['ATAC_spectral_harmony'] = embedding

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_harmony_multi.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_harmony_multi'] = embedding

In [ ]:
embedding = pd.read_csv('processed/multi_spectral/joint/embedding_raw.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral'] = embedding

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_multi_harmony.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_multi_harmony'] = embedding'

embedding = pd.read_csv('processed/multi_spectral/joint/embedding_cellhint.csv', index_col=0).to_numpy()
rna.obsm['joint_spectral_multi_cellhint'] = embedding

embedding = pd.read_csv('../7_horizontal_RNA_ATAC/results/blood/uinmf_latent_k30.csv', header=0, index_col=0).to_numpy()
rna.obsm['joint_uinmf'] = embedding

embedding = pd.read_csv('../7_horizontal_RNA_ATAC/results/blood/latent_mtv.csv', header=None).to_numpy()
rna.obsm['joint_multivi'] = embedding

In [ ]:
embedding = pd.read_csv('processed/pca/rna_pca.csv', index_col=0).to_numpy()
rna.obsm['RNA_pca'] = embedding

embedding = pd.read_csv('processed/pca/rna_pca_harmony.csv', index_col=0).to_numpy()
rna.obsm['RNA_pca_harmony'] = embedding

embedding = pd.read_csv('processed/lsi/atac_lsi.csv', index_col=0).to_numpy()
rna.obsm['ATAC_lsi'] = embedding

embedding = pd.read_csv('processed/lsi/atac_lsi_harmony.csv', index_col=0).to_numpy()
rna.obsm['ATAC_lsi_harmony'] = embedding

# umap = pd.read_csv('./processed/wnn/umap_m.csv').to_numpy()
# rna.obsm['joint_wnn'] = umap

# umap = pd.read_csv('./processed/wnn/umap_h_m.csv').to_numpy()
# rna.obsm['joint_harmony_wnn'] = umap

In [ ]:
bm = Benchmarker(
    rna,
    batch_key="sample",
    label_key="level_3",
    embedding_obsm_keys=['RNA_spectral','RNA_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

In [ ]:
bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_results_table.pdf", dpi=300)

In [ ]:
df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_results.csv", index=True)

In [ ]:
# Save the Benchmarker object
with open("benchmarker.pkl", "wb") as f:
    pickle.dump(bm, f)

# Load the Benchmarker object
with open("benchmarker.pkl", "rb") as f:
    bm_loaded = pickle.load(f)